<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase2/phase2_02_video_frame_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Sector 2: Configuration, Storage Setup, and Raw Data Ingestion

This notebook initializes the reproducible Phase 2 configuration, mounts the
persistent Google Drive storage, and prepares the Data Lake directory structure.
It checks whether Kvasir-Capsule is already available at the configured storage
location and downloads it from the official source only when it is missing.
The dataset contents are then inventoried and validated before downstream
processing begins. This notebook contains the full instructions that handle the dataset provisioning, normalize the fields, perform the validation and output usuable data for the next Sectors of the Data Lake.

### 1. Install dependencies and imports

In [1]:
%pip install -q \
    pandas \
    numpy \
    pyarrow \
    opencv-python-headless \
    scikit-image \
    scikit-learn \
    iterative-stratification \
    tqdm \
    gdown

In [2]:
from pathlib import Path
from collections import defaultdict

import re
import mimetypes
import os
import json
import random
import warnings
import subprocess
import shutil
import sys
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from IPython.display import display
from collections import deque
from uuid import uuid4
from contextlib import contextmanager


import filecmp
import lzma
import stat
import tempfile
import cv2
import numpy as np
import pandas as pd
import torch
import zipfile
import tarfile
import zlib
import gzip
import ast
import hashlib


from tqdm.auto import tqdm

from skimage.metrics import structural_similarity as ssim

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)

### 2. Github Colab Sync

In [3]:
GIT_CONFIG = {
    "repo_url": "https://github.com/bogdanparvu18/msc-graduate-project.git",
    "branch": "main",
    "local_repo_dir": "/content/msc-graduate-project",
    "config_notebook_name": "phase2_01_ingestion.ipynb",
    "repo_output_dir": "outputs/phase2",
    "config_output_filename": "phase2_01_ingestion_{run_id}_config.json",
    "output_groups": ["configs", "results", "reports"],
    "dataset_audit_subdir": "dataset_audit",
    "dataset_audit_excluded_artifacts": [
        "physical_image_inventory",
        "metadata_clean",
    ],
    "allowed_extensions": [".json", ".csv", ".md", ".txt"],
    "max_file_size_mib": 20,
    "token_secret_name": "GITHUB_TOKEN",
    "github_username": "bogdanparvu18",
    "author_name": "Bogdan Parvu",
    "author_email": "bparvu@lakeheadu.ca",
    "commit_message": "Phase 2: update ingestion config and audit outputs",
    "confirm_push": True,
}


def run_git(repo, *args, env=None, input_text=None, allowed_codes=(0,)):
    """Runs Git without a shell; never embeds credentials in arguments."""
    command = ["git"] + (["-C", str(repo)] if repo is not None else [])
    process_env = {
        k: v for k, v in os.environ.items()
        if not k.startswith("GIT_TRACE") and k != "GIT_CURL_VERBOSE"
    }
    process_env.update({"GIT_TERMINAL_PROMPT": "0", **(env or {})})
    result = subprocess.run(
        command + list(args), input=input_text, env=process_env,
        text=True, capture_output=True,
    )
    if result.returncode not in allowed_codes:
        message = (result.stderr or result.stdout).strip()
        token = process_env.get("GITHUB_TOKEN")
        if token:
            message = message.replace(token, "[REDACTED]")
        raise RuntimeError(f"Git failed:\n{message}")
    return result.stdout


def prepare_git_repository(settings):
    """Clones once or fast-forwards a clean, matching local repository."""
    repo = Path(settings["local_repo_dir"]).expanduser().resolve()
    branch = settings["branch"]
    if not (repo / ".git").is_dir():
        if repo.exists() and (not repo.is_dir() or any(repo.iterdir())):
            raise RuntimeError(f"Destination exists but is not an empty Git checkout: {repo}")
        repo.parent.mkdir(parents=True, exist_ok=True)
        run_git(None, "clone", "--branch", branch, "--single-branch",
                settings["repo_url"], str(repo))

    expected = settings["repo_url"].rstrip("/").removesuffix(".git")
    for options in [("get-url",), ("get-url", "--push")]:
        actual = run_git(repo, "remote", *options, "origin").strip()
        if actual.rstrip("/").removesuffix(".git") != expected:
            raise RuntimeError("Origin does not match GIT_CONFIG. Nothing was pushed.")
    if run_git(repo, "branch", "--show-current").strip() != branch:
        raise RuntimeError(f"Expected branch {branch}; no automatic branch switching.")
    if run_git(repo, "status", "--porcelain").strip():
        raise RuntimeError("Local checkout has changes. Resolve them before synchronizing; no reset is performed.")

    run_git(repo, "pull", "--ff-only", "origin", branch)
    if run_git(repo, "rev-list", f"origin/{branch}..HEAD").strip():
        raise RuntimeError("There are unpublished local commits. Resolve/push them before starting another sync.")
    return repo


def load_config_from_notebook(repo, notebook_name):
    """Reads the literal CONFIG dictionary; does NOT execute the notebook."""
    tracked = run_git(repo, "ls-files", "-z").split("\0")
    matches = [repo / p for p in tracked if p and Path(p).name == notebook_name]
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one tracked {notebook_name}; found: {matches}")
    notebook_path = matches[0]
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    configs = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", "")
        source = "".join(source) if isinstance(source, list) else source
        try:
            statements = ast.parse(source).body
        except SyntaxError:
            if re.search(r"(?m)^\s*CONFIG\s*=", source):
                raise ValueError("Keep CONFIG = {...} in a plain Python cell without shell/magic commands.") from None
            continue
        for node in statements:
            targets = node.targets if isinstance(node, ast.Assign) else (
                [node.target] if isinstance(node, ast.AnnAssign) else []
            )
            if not any(isinstance(t, ast.Name) and t.id == "CONFIG" for t in targets):
                continue
            if not isinstance(node.value, ast.Dict):
                continue  # Ignore calls such as CONFIG = load_config(...).
            try:
                configs.append(ast.literal_eval(node.value))
            except (ValueError, TypeError, SyntaxError):
                raise ValueError("CONFIG must contain literal values, not calls, external variables, or **CONFIG.") from None
    if len(configs) != 1:
        raise ValueError(f"Expected one literal CONFIG dictionary; found {len(configs)}.")
    json.dumps(configs[0], allow_nan=False)  # Check JSON compatibility.
    return configs[0], notebook_path


def file_sha256(path):
    """Hashes file contents without loading the entire file into RAM."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def collect_publishable_outputs(dirs, repo, settings):
    """Builds a publication plan from output directories only."""
    relative_root = Path(settings["repo_output_dir"])
    if relative_root.is_absolute() or ".." in relative_root.parts:
        raise ValueError("repo_output_dir must be a repository-relative path.")
    records, skipped = [], []
    for group in settings["output_groups"]:
        if group not in {"configs", "results", "reports"}:
            raise ValueError(f"Unsupported output group: {group}")
        source_dir = Path(dirs[f"{group}_dir"]).resolve()
        if not source_dir.is_dir():
            raise FileNotFoundError(f"Output directory is missing: {source_dir}")
        for source in sorted(source_dir.rglob("*")):
            if not source.is_file():
                continue
            relative = source.relative_to(source_dir)
            if any(part.startswith(".") for part in relative.parts) or source.suffix.lower() not in settings["allowed_extensions"]:
                skipped.append(f"{group}/{relative.as_posix()}")
                continue
            if source.is_symlink() or not source.resolve().is_relative_to(source_dir):
                raise ValueError(f"Refusing a symlink/out-of-scope source: {source}")
            size = source.stat().st_size
            if size > settings["max_file_size_mib"] * 1024 ** 2:
                raise ValueError(f"File exceeds publication size limit: {source}")
            repo_relative = relative_root / group / relative
            destination = repo / repo_relative
            if destination.is_symlink() or not destination.resolve().is_relative_to(repo):
                raise ValueError(f"Unsafe destination: {destination}")
            digest = file_sha256(source)
            records.append({
                "source": source, "destination": destination,
                "repo_relative": repo_relative.as_posix(), "group": group,
                "size": size, "sha256": digest,
                "changed": not destination.exists() or file_sha256(destination) != digest,
            })
    return records, skipped


def build_dataset_audit_github_files(
    storage_spec, catalog_filename, excluded_artifacts,
):
    """Derives audit filenames from the specification used by the exporter."""
    missing_columns = sorted({"artifact", "format"} - set(storage_spec.columns))
    if missing_columns:
        raise KeyError(f"Audit storage specification is missing: {missing_columns}")

    selected = (
        storage_spec.loc[
            ~storage_spec["artifact"].isin(excluded_artifacts),
            ["artifact", "format"],
        ]
        .astype("string")
        .assign(filename=lambda data: data["artifact"] + "." + data["format"])
    )
    # The exporter writes this catalog separately from AUDIT_STORAGE_SPEC.
    filenames = pd.concat(
        [selected["filename"], pd.Series([catalog_filename], dtype="string")],
        ignore_index=True,
    )
    valid = filenames.str.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_.-]*", na=False)
    if not valid.all():
        raise ValueError(
            "Audit exports must have nonempty, simple filenames: "
            f"{filenames.loc[~valid].tolist()}"
        )
    return tuple(filenames.drop_duplicates().sort_values())


def collect_publishable_dataset_audit(
    dirs, repo, settings, storage_spec, catalog_filename,
):
    """Plans audit publication; existing remote files are never updated.

    Call after prepare_git_repository(), so HEAD matches the remote branch.
    This function inspects files but does not copy, stage, commit or push.
    """
    subdir = Path(settings["dataset_audit_subdir"])
    if subdir.is_absolute() or ".." in subdir.parts or subdir == Path("."):
        raise ValueError("dataset_audit_subdir must be a nonempty relative path.")
    relative_root = Path(settings["repo_output_dir"])
    if relative_root.is_absolute() or ".." in relative_root.parts:
        raise ValueError("repo_output_dir must be a repository-relative path.")

    source_dir = (Path(dirs["curated_data_dir"]) / subdir).resolve()
    destination_root = relative_root / subdir
    excluded_artifacts = settings["dataset_audit_excluded_artifacts"]
    filenames = build_dataset_audit_github_files(
        storage_spec, catalog_filename, excluded_artifacts,
    )
    excluded_spec = storage_spec.loc[
        storage_spec["artifact"].isin(excluded_artifacts), ["artifact", "format"]
    ].astype("string")
    skipped = [
        f"{subdir.as_posix()}/{name}"
        for name in (excluded_spec["artifact"] + "." + excluded_spec["format"])
    ]
    allowed_extensions = {extension.lower() for extension in settings["allowed_extensions"]}
    tracked_files = set(filter(None, run_git(
        repo, "--literal-pathspecs", "ls-tree", "-r", "--name-only", "-z",
        "HEAD", "--", destination_root.as_posix(),
    ).split("\0")))

    records = []
    for filename in filenames:
        if Path(filename).suffix.lower() not in allowed_extensions:
            skipped.append(f"{subdir.as_posix()}/{filename}")
            continue
        source = source_dir / filename
        repo_relative = (destination_root / filename).as_posix()
        destination = repo / repo_relative
        if destination.is_symlink() or not destination.resolve().is_relative_to(repo):
            raise ValueError(f"Unsafe destination: {destination}")

        already_on_github = repo_relative in tracked_files
        size, digest = None, None
        if not already_on_github:
            if not source.is_file():
                raise FileNotFoundError(
                    f"Audit report is absent from GitHub and Drive: {source}. "
                    "Run the dataset audit export cell first."
                )
            if source.is_symlink() or not source.resolve().is_relative_to(source_dir):
                raise ValueError(f"Refusing a symlink/out-of-scope source: {source}")
            size = source.stat().st_size
            if size > settings["max_file_size_mib"] * 1024 ** 2:
                raise ValueError(f"File exceeds publication size limit: {source}")
            digest = file_sha256(source)

        records.append({
            "source": source, "destination": destination,
            "repo_relative": repo_relative, "group": "dataset_audit",
            "size": size, "sha256": digest,
            "changed": not already_on_github,
        })
    return records, skipped


def authenticated_push(repo, settings, token):
    """Passes the token via an ephemeral environment, not the remote URL."""
    with tempfile.TemporaryDirectory(prefix="mmvqa-git-") as folder:
        askpass = Path(folder) / "askpass.sh"
        askpass.write_text(
            '#!/bin/sh\ncase "$1" in\n'
            '*Username*) printf "%s\\n" "$MMVQA_GIT_USER" ;;\n'
            '*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
            'esac\n', encoding="utf-8",
        )
        askpass.chmod(0o700)
        environment = {
            "GIT_ASKPASS": str(askpass),
            "MMVQA_GIT_USER": settings["github_username"],
            "GITHUB_TOKEN": token,
            "LC_ALL": "C",
        }
        run_git(repo, "-c", "credential.helper=", "push", "origin",
                f"HEAD:refs/heads/{settings['branch']}", env=environment)


def publish_phase2_outputs(config, dirs, settings):
    """Publishes runtime CONFIG, saved outputs and missing dataset audit files.

    Regular outputs are updated when their content changes. Audit filenames
    follow AUDIT_STORAGE_SPEC; audit files already on GitHub are preserved.
    All selected changes share the existing confirmation, commit and push.
    """
    if config["storage_backend"] == "google_drive" and not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Mount Google Drive before publishing.")
    expected_root = Path(config["output_dir"])
    if not expected_root.is_absolute():
        expected_root = Path(config["storage_root"]) / expected_root
    expected_root = expected_root.resolve()
    for group in settings["output_groups"]:
        if Path(dirs[f"{group}_dir"]).resolve() != expected_root / group:
            raise ValueError("DIRS and CONFIG point to different output locations.")
        if not Path(dirs[f"{group}_dir"]).is_dir():
            raise FileNotFoundError(f"Run prepare_phase2_dirs first: {group}")

    # Audit definitions are needed only here, not during the bootstrap below.
    missing_audit_definitions = [
        name for name in ("AUDIT_STORAGE_SPEC", "AUDIT_CATALOG_FILENAME")
        if name not in globals()
    ]
    if missing_audit_definitions:
        raise RuntimeError(
            "Run the dataset audit definition/export cell before publishing. "
            f"Missing: {missing_audit_definitions}"
        )

    repo = prepare_git_repository(settings)  # Refresh before copying, not afterwards.
    run_id = config.get("run_id")
    if not isinstance(run_id, str) or not run_id.strip():
        raise ValueError("Initialize CONFIG['run_id'] at the start of the run before publishing.")
    config_filename = settings["config_output_filename"].format(run_id=run_id)
    if Path(config_filename).name != config_filename or not config_filename.endswith(".json"):
        raise ValueError("config_output_filename must resolve to a JSON filename, not a path.")
    config_path = Path(dirs["configs_dir"]) / config_filename
    config_text = json.dumps(config, indent=2, ensure_ascii=False, sort_keys=True, allow_nan=False) + "\n"
    config_path.write_text(config_text, encoding="utf-8")
    records, skipped = collect_publishable_outputs(dirs, repo, settings)
    audit_records, audit_skipped = collect_publishable_dataset_audit(
        dirs=dirs, repo=repo, settings=settings,
        storage_spec=AUDIT_STORAGE_SPEC,
        catalog_filename=AUDIT_CATALOG_FILENAME,
    )
    records = [*records, *audit_records]
    skipped = [*skipped, *audit_skipped]
    planned_paths = pd.Series([r["repo_relative"] for r in records], dtype="string")
    if planned_paths.duplicated().any():
        raise ValueError("Multiple publication sources target the same repository path.")
    if not any(r["group"] in {"reports", "results", "dataset_audit"} for r in records):
        raise RuntimeError("No publishable reports/results found. Save them to Drive first.")
    audit_publication = {
        "already_on_github": [r["repo_relative"] for r in audit_records if not r["changed"]],
        "published": [],
    }
    changed = [r for r in records if r["changed"]]
    print(f"Eligible: {len(records)} | Changed: {len(changed)} | Excluded: {len(skipped)}")
    print(f"Dataset audit already on GitHub: {len(audit_publication['already_on_github'])}")
    for path in audit_publication["already_on_github"]:
        print("ALREADY ON GITHUB:", path)
    for path in skipped:
        print("EXCLUDED:", path)
    if not changed:
        print("No files need publication under the configured policies. No commit or push is needed.")
        return {"status": "unchanged", "published": 0, "excluded": skipped,
                "dataset_audit": audit_publication}

    paths = [r["repo_relative"] for r in changed]
    path_input = "\0".join(paths) + "\0"
    ignored = run_git(repo, "check-ignore", "--stdin", "-z",
                      input_text=path_input, allowed_codes=(0, 1))
    if ignored:
        raise RuntimeError("Adjust .gitignore for these outputs first: " + ", ".join(filter(None, ignored.split("\0"))))
    for record in changed:
        print(f"PUBLISH: {record['repo_relative']} ({record['size'] / 1024:.1f} KiB)")
    if settings["confirm_push"] and input("Publish these files? Type PUSH: ").strip() != "PUSH":
        return {"status": "cancelled", "published": 0, "excluded": skipped,
                "dataset_audit": audit_publication}

    from google.colab import userdata
    token = userdata.get(settings["token_secret_name"]).strip()
    if not token:
        raise ValueError("The GitHub token is empty.")
    email = settings["author_email"].strip() or input("Git commit email (GitHub/noreply): ").strip()
    if "@" not in email:
        raise ValueError("Provide your GitHub commit email or your exact GitHub noreply address.")
    # Detect changes made while the user was reviewing the publication plan.
    if any(file_sha256(r["source"]) != r["sha256"] for r in changed):
        raise RuntimeError("An output changed during review. Run publication again.")
    if any(token.encode() in r["source"].read_bytes() for r in changed):
        raise ValueError("An output contains the GitHub token. Publication stopped.")
    for record in changed:
        record["destination"].parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(record["source"], record["destination"])
    run_git(repo, "--literal-pathspecs", "add", "--pathspec-from-file=-",
            "--pathspec-file-nul", input_text=path_input)
    staged = set(filter(None, run_git(repo, "diff", "--cached", "--name-only", "-z").split("\0")))
    if staged - set(paths):
        raise RuntimeError("Unexpected staged files. Inspect the local checkout before proceeding.")
    if not staged:
        print("No changes remain after Git text normalization.")
        return {"status": "unchanged", "published": 0, "excluded": skipped,
                "dataset_audit": audit_publication}
    print(run_git(repo, "diff", "--cached", "--stat"))
    run_git(repo, "-c", f"user.name={settings['author_name']}", "-c", f"user.email={email}",
            "commit", "-m", settings["commit_message"])
    authenticated_push(repo, settings, token)
    commit = run_git(repo, "rev-parse", "HEAD").strip()
    audit_publication["published"] = sorted(
        staged & {r["repo_relative"] for r in audit_records}
    )
    print(f"Pushed {len(staged)} files to {settings['branch']}. Commit: {commit}")
    return {"status": "pushed", "published": len(staged), "commit": commit,
            "excluded": skipped, "dataset_audit": audit_publication}


# Bootstrap only: no dataset download and no report generation.
REPO_DIR = prepare_git_repository(GIT_CONFIG)

CONFIG, CONFIG_NOTEBOOK_PATH = load_config_from_notebook(
    repo=REPO_DIR,
    notebook_name=GIT_CONFIG["config_notebook_name"],
)

# Initialize ONCE, before running the pipeline, not inside publication.
# Literal defaults maintain compatibility with existing repository CONFIGs.
# Defining these two keys in the source CONFIG overrides these defaults.
CONFIG.setdefault("timezone", "America/Toronto")
CONFIG.setdefault("run_id_format", "%Y%m%d_%H%M%S")

RUN_ID = datetime.now(
    ZoneInfo(CONFIG["timezone"])
).strftime(CONFIG["run_id_format"])

CONFIG["run_id"] = RUN_ID

CONFIG_SOURCE_COMMIT = run_git(REPO_DIR, "rev-parse", "HEAD").strip()

print("Repository:", REPO_DIR)
print("CONFIG loaded from:", CONFIG_NOTEBOOK_PATH.relative_to(REPO_DIR))
print("Source commit:", CONFIG_SOURCE_COMMIT)
print("Run ID:", CONFIG["run_id"])
print("Run the existing DIRS, ingestion, validation, and report-saving cells next.")

Repository: /content/msc-graduate-project
CONFIG loaded from: notebooks/phase2/phase2_01_ingestion.ipynb
Source commit: a15e15daa30da6debf073eaca1c04aeb11598e52
Run ID: 20260923_124632
Run the existing DIRS, ingestion, validation, and report-saving cells next.


### 3. Load declarative configuration and reproducibility

In [4]:
CONFIG = {
    # --------------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------------

    "seed": 42,

    # --------------------------------------------------------------
    # Persistent storage
    # --------------------------------------------------------------

    "storage_backend": "google_drive",

    "storage_root": (
        "/content/drive/MyDrive/"
        "MMVQA_Clinical"
    ),

    "raw_data_dir": (
        "data/raw/kvasir_capsule"
    ),

    "interim_data_dir": (
        "data/interim/phase2"
    ),

    "curated_data_dir": (
        "data/curated/phase2"
    ),

    "output_dir": (
        "outputs/phase2"
    ),

    # --------------------------------------------------------------
    # Raw-dataset acquisition
    # --------------------------------------------------------------

    "dataset_download_enabled": True,

    "dataset_google_drive_folder_url": (
        "https://drive.google.com/drive/folders/"
        "18vEHN1CG7oNFKdT2NmhtJjrFhb3tLG1Z"
    ),

    "dataset_archive_repair_enabled": True,

    # --------------------------------------------------------------
    # Raw-dataset validation
    # --------------------------------------------------------------

    "dataset_validation": {
        "osf_storage_subdir": (
            "osfstorage"
        ),

        "required_metadata_file": (
            "metadata.csv"
        ),

        "labelled_images_subdir": (
            "labelled_images"
        ),

        "minimum_video_files": 117,

        "minimum_labelled_images": 47238,

        "image_extensions": [
            ".png",
            ".jpg",
            ".jpeg",
        ],

        "video_extensions": [
            ".avi",
            ".mp4",
            ".mkv",
        ],
    },


    # --------------------------------------------------------------
    # Normalized clinical taxonomy
    # --------------------------------------------------------------

    # Keys must match finding_class_normalized.
    "clinical_group_map": {
        "ampulla of vater": (
            "anatomical_landmark"
        ),

        "ileocecal valve": (
            "anatomical_landmark"
        ),

        "pylorus": (
            "anatomical_landmark"
        ),

        "normal clean mucosa": (
            "normal_mucosa"
        ),

        "reduced mucosal view": (
            "visibility_limitation"
        ),

        "blood fresh": "bleeding",

        "blood hematin": "bleeding",

        "angiectasia": (
            "vascular_lesion"
        ),

        "erosion": (
            "mucosal_lesion"
        ),

        "erythema": (
            "mucosal_lesion"
        ),

        "ulcer": (
            "mucosal_lesion"
        ),

        "lymphangiectasia": (
            "lymphatic_lesion"
        ),

        "polyp": (
            "protruding_lesion"
        ),

        "foreign body": (
            "foreign_body"
        ),
    },
}


CONFIG

{'seed': 42,
 'storage_backend': 'google_drive',
 'storage_root': '/content/drive/MyDrive/MMVQA_Clinical',
 'raw_data_dir': 'data/raw/kvasir_capsule',
 'interim_data_dir': 'data/interim/phase2',
 'curated_data_dir': 'data/curated/phase2',
 'output_dir': 'outputs/phase2',
 'dataset_download_enabled': True,
 'dataset_google_drive_folder_url': 'https://drive.google.com/drive/folders/18vEHN1CG7oNFKdT2NmhtJjrFhb3tLG1Z',
 'dataset_archive_repair_enabled': True,
 'dataset_validation': {'osf_storage_subdir': 'osfstorage',
  'required_metadata_file': 'metadata.csv',
  'labelled_images_subdir': 'labelled_images',
  'minimum_video_files': 117,
  'minimum_labelled_images': 47238,
  'image_extensions': ['.png', '.jpg', '.jpeg'],
  'video_extensions': ['.avi', '.mp4', '.mkv']},
 'clinical_group_map': {'ampulla of vater': 'anatomical_landmark',
  'ileocecal valve': 'anatomical_landmark',
  'pylorus': 'anatomical_landmark',
  'normal clean mucosa': 'normal_mucosa',
  'reduced mucosal view': 'visibilit

### 4. Mount Google Drive Storage Backend

In [5]:
def mount_storage(config):
    """
    Mounts persistent storage when required by the configured backend.

    For Google Colab + Google Drive, this mounts Drive under /content/drive.
    It does not download or copy the dataset.
    """

    storage_backend = config["storage_backend"]

    if storage_backend == "google_drive":

        try:
            from google.colab import drive

            drive.mount(
                "/content/drive",
                force_remount=False,
            )

            print("Google Drive mounted.")

        except ImportError:
            raise RuntimeError(
                "Google Drive backend is configured, "
                "but the notebook is not running in Google Colab."
            )

    elif storage_backend == "local":

        print("Using local storage.")

    else:

        raise ValueError(
            f"Unsupported storage backend: {storage_backend}"
        )


mount_storage(CONFIG)

Mounted at /content/drive
Google Drive mounted.


### 5. Define data paths

In [6]:
def prepare_phase2_dirs(config):
    """
    Resolves and creates the directory structure required
    for Phase 2.

    Main directories are declared in CONFIG. Relative paths
    are resolved against storage_root, while absolute paths
    are preserved.

    Raw-dataset reference paths are returned, but dataset
    source files and source subdirectories are not created.

    Side effect:
        Creates writable Phase 2 directories on persistent
        storage.

    Returns:
        Dictionary containing resolved Path objects.
    """

    storage_root = Path(
        config["storage_root"]
    )

    validation = config[
        "dataset_validation"
    ]


    def resolve_path(path_value):
        """
        Resolves a CONFIG path against storage_root.

        Absolute paths are returned unchanged.
        """

        path = Path(path_value)

        if path.is_absolute():
            return path

        return storage_root / path


    # --------------------------------------------------------------
    # Main CONFIG directories
    # --------------------------------------------------------------

    raw_data_dir = resolve_path(
        config["raw_data_dir"]
    )

    interim_data_dir = resolve_path(
        config["interim_data_dir"]
    )

    curated_data_dir = resolve_path(
        config["curated_data_dir"]
    )

    output_dir = resolve_path(
        config["output_dir"]
    )


    # --------------------------------------------------------------
    # Raw-dataset source references
    # --------------------------------------------------------------

    dataset_root_dir = (
        raw_data_dir
        / validation["osf_storage_subdir"]
    )

    labelled_images_dir = (
        dataset_root_dir
        / validation["labelled_images_subdir"]
    )

    metadata_path = (
        dataset_root_dir
        / validation["required_metadata_file"]
    )


    # --------------------------------------------------------------
    # Writable directories created by the pipeline
    # --------------------------------------------------------------

    created_directories = {
        # Main directories
        "raw_data_dir":
            raw_data_dir,

        "interim_data_dir":
            interim_data_dir,

        "curated_data_dir":
            curated_data_dir,

        "output_dir":
            output_dir,

        # Derived data directories
        "temporal_frames_dir":
            interim_data_dir
            / "temporal_frames",

        "manifests_dir":
            curated_data_dir
            / "manifests",

        "splits_dir":
            curated_data_dir
            / "splits",

        # Derived output directories
        "configs_dir":
            output_dir
            / "configs",

        "results_dir":
            output_dir
            / "results",

        "reports_dir":
            output_dir
            / "reports",
    }


    # --------------------------------------------------------------
    # Create only writable pipeline directories
    # --------------------------------------------------------------

    for directory in created_directories.values():
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )


    # --------------------------------------------------------------
    # Dataset paths that must come from the OSF dataset
    # --------------------------------------------------------------

    dataset_paths = {
        "dataset_root_dir":
            dataset_root_dir,

        "labelled_images_dir":
            labelled_images_dir,

        "metadata_path":
            metadata_path,
    }


    return {
        **created_directories,
        **dataset_paths,
    }


DIRS = prepare_phase2_dirs(
    CONFIG
)

DIRS

{'raw_data_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/raw/kvasir_capsule'),
 'interim_data_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/interim/phase2'),
 'curated_data_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2'),
 'output_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/outputs/phase2'),
 'temporal_frames_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/interim/phase2/temporal_frames'),
 'manifests_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2/manifests'),
 'splits_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2/splits'),
 'configs_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/configs'),
 'results_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/results'),
 'reports_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/reports'),
 'dataset_root_dir': PosixPath('/content/drive/MyDrive/MMVQA